# Baseline FAYAM — TimeSeriesTransformer (4 clusters locaux)

Reproduction du modèle Transformer de la baseline FAYAM pour la prédiction de charge FaaS, **sur nos 4 clusters locaux** (C0, C4, C6, C8 — 19 fonctions).

| Champ | Valeur |
|-------|--------|
| **Run** | `2026-05-05_baseline-fayam-local-clusters` |
| **Phase** | Phase 1 — Reproduction baseline (sur clusters locaux) |
| **Seed** | 998 |
| **Epochs** | 51 |
| **Dataset** | 4 CSV locaux : C0 (3 fct), C4 (5 fct), C6 (5 fct), C8 (6 fct) — 19 séries × 20 160 pas |
| **Source** | `Drive/Recherche/Datasets/cluster_{0,4,6,8}.csv` (Colab) ou `memoire/06-datasets/raw/` (local) |

**Métriques cibles** : MASE, sMAPE, RMSE, R², Spearman — **par série ET par cluster**  
**Extra** : extraction `output_attentions=True` post-entraînement (prépare H1/H3)

> Avant de lancer : Runtime → Change runtime type → **T4 GPU**

## 1 — Setup

In [ ]:
import subprocess
gpu = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(gpu.stdout if gpu.returncode == 0 else 'Pas de GPU — vérifier le runtime.')

In [ ]:
%%capture
!pip install -q transformers datasets "gluonts[torch]" accelerate evaluate scipy scikit-learn tqdm openpyxl ujson

## 2 — Imports, seed et configuration

In [ ]:
import random
import os
import json
import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from functools import lru_cache, partial
from typing import Optional, Iterable
from tqdm.notebook import tqdm

# ── Seed ────────────────────────────────────────────────────────────────────
SEED = 998
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f'PyTorch   : {torch.__version__}')
print(f'CUDA dispo : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU       : {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Hyperparamètres (source unique — modifier ici uniquement) ────────────────
FREQ               = '1T'
PREDICTION_LENGTH  = 120
CONTEXT_LENGTH     = PREDICTION_LENGTH * 2   # 240
ENCODER_LAYERS     = 4
DECODER_LAYERS     = 4
D_MODEL            = 32
EMBEDDING_DIM      = [2]

BATCH_SIZE_TRAIN   = 256
BATCH_SIZE_TEST    = 64
NUM_BATCHES_EPOCH  = 100
NUM_EPOCHS         = 51
LR                 = 6e-4
BETAS              = (0.9, 0.95)
WEIGHT_DECAY       = 1e-1
GRAD_CLIP_NORM     = 1.0

CHECKPOINT_EVERY   = 10
MAX_ATTN_SAMPLES   = 100  # plafond — tronqué automatiquement si dataset plus petit

# Nouveau RUN_NAME (run sur clusters locaux ≠ run sur FaalSa/dataME)
RUN_NAME = '2026-05-05_baseline-fayam-local-clusters'
print('Config OK')

## 3 — Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE = f'/content/drive/MyDrive/m2-xai-faas/experiments/{RUN_NAME}'
for subdir in ['checkpoints', 'results', 'attentions']:
    os.makedirs(f'{DRIVE_BASE}/{subdir}', exist_ok=True)
print(f'Dossier run : {DRIVE_BASE}')

## 4 — Chargement du dataset

In [ ]:
from pathlib import Path
from datasets import Dataset

# DATA_DIR adaptatif Colab/local
try:
    from google.colab import drive  # noqa: F401
    DATA_DIR = Path('/content/drive/MyDrive/Recherche/Datasets')
except ImportError:
    DATA_DIR = Path('../../memoire/06-datasets/raw').resolve()

CLUSTER_IDS = [0, 4, 6, 8]
START_DATE  = '2021-01-01 00:00:00'

# Chargement des 4 CSV : 1 ligne = 1 fonction, colonnes 1..20160 = pas de temps
all_series = []
for cid in CLUSTER_IDS:
    df = pd.read_csv(DATA_DIR / f'cluster_{cid}.csv', index_col='Function')
    for func_id, row in df.iterrows():
        all_series.append({
            'function_id': int(func_id),
            'cluster':     cid,
            'target_full': row.values.astype(np.float32),
        })

assert len(all_series) == 19, f'Attendu 19 séries, obtenu {len(all_series)}'

# Convention FAYAM :
#   train = série tronquée des PREDICTION_LENGTH derniers points
#   test  = série complète (le ValidationSplitSampler extrait les 120 derniers)
train_rows, test_rows = [], []
for ts_idx, s in enumerate(all_series):
    base = {
        'start':           START_DATE,
        'feat_static_cat': [ts_idx],
        'cluster':         s['cluster'],
        'function_id':     s['function_id'],
    }
    train_rows.append({**base, 'target': s['target_full'][:-PREDICTION_LENGTH].tolist()})
    test_rows.append({**base, 'target': s['target_full'].tolist()})

train_dataset = Dataset.from_list(train_rows)
test_dataset  = Dataset.from_list(test_rows)

# Sanity check (validation_example construit pour l'assertion + cellule de plot suivante)
train_example      = train_dataset[0]
validation_example = {'target': all_series[0]['target_full']}
assert len(train_example['target']) + PREDICTION_LENGTH == len(validation_example['target']), \
    'Assertion longueur échouée : len(train) + prediction_length != len(full series)'

print(f'DATA_DIR : {DATA_DIR}')
print(f'Séries train : {len(train_dataset)}')
print(f'Séries test  : {len(test_dataset)}')
print(f'Longueur série 0 (train) : {len(train_example["target"])}')
print(f'Longueur série 0 (full)  : {len(all_series[0]["target_full"])}')

counts = {cid: sum(1 for s in all_series if s['cluster'] == cid) for cid in CLUSTER_IDS}
print(f'Répartition par cluster : {counts}')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
ax.plot(train_example['target'], color='steelblue', linewidth=0.5, label='Train')
ax.plot(validation_example['target'], color='tomato', alpha=0.5, linewidth=0.5, label='Validation')
ax.set_title('Série 0 — train (bleu) vs validation (rouge)')
ax.legend()
plt.tight_layout()
plt.show()

## 5 — Pipeline GluonTS

In [ ]:
from gluonts.time_feature import (
    get_lags_for_frequency,
    time_features_from_frequency_str,
)
from gluonts.dataset.field_names import FieldName
from gluonts.transform import (
    AddAgeFeature, AddObservedValuesIndicator, AddTimeFeatures,
    AsNumpyArray, Chain, ExpectedNumInstanceSampler, InstanceSplitter,
    RemoveFields, TestSplitSampler, Transformation,
    ValidationSplitSampler, VstackFeatures, RenameFields,
)
from gluonts.transform.sampler import InstanceSampler
from gluonts.itertools import Cyclic, Cached
from gluonts.dataset.loader import as_stacked_batches
from transformers import PretrainedConfig

lags_sequence = get_lags_for_frequency(FREQ)
time_features  = time_features_from_frequency_str(FREQ)

print(f'Lags sequence length : {len(lags_sequence)}')
print(f'Time features count  : {len(time_features)}')


@lru_cache(10_000)
def convert_to_pandas_period(date, freq):
    return pd.Period(date, freq)


def transform_start_field(batch, freq):
    batch['start'] = [convert_to_pandas_period(date, freq) for date in batch['start']]
    return batch


train_dataset.set_transform(partial(transform_start_field, freq=FREQ))
test_dataset.set_transform(partial(transform_start_field, freq=FREQ))

In [ ]:
def create_transformation(freq: str, config: PretrainedConfig) -> Transformation:
    remove_field_names = []
    if config.num_static_real_features == 0:
        remove_field_names.append(FieldName.FEAT_STATIC_REAL)
    if config.num_dynamic_real_features == 0:
        remove_field_names.append(FieldName.FEAT_DYNAMIC_REAL)
    if config.num_static_categorical_features == 0:
        remove_field_names.append(FieldName.FEAT_STATIC_CAT)

    return Chain(
        [RemoveFields(field_names=remove_field_names)]
        + ([AsNumpyArray(field=FieldName.FEAT_STATIC_CAT, expected_ndim=1, dtype=int)]
           if config.num_static_categorical_features > 0 else [])
        + ([AsNumpyArray(field=FieldName.FEAT_STATIC_REAL, expected_ndim=1)]
           if config.num_static_real_features > 0 else [])
        + [
            AsNumpyArray(
                field=FieldName.TARGET,
                expected_ndim=1 if config.input_size == 1 else 2,
            ),
            AddObservedValuesIndicator(
                target_field=FieldName.TARGET,
                output_field=FieldName.OBSERVED_VALUES,
            ),
            AddTimeFeatures(
                start_field=FieldName.START,
                target_field=FieldName.TARGET,
                output_field=FieldName.FEAT_TIME,
                time_features=time_features_from_frequency_str(freq),
                pred_length=config.prediction_length,
            ),
            AddAgeFeature(
                target_field=FieldName.TARGET,
                output_field=FieldName.FEAT_AGE,
                pred_length=config.prediction_length,
                log_scale=True,
            ),
            VstackFeatures(
                output_field=FieldName.FEAT_TIME,
                input_fields=[FieldName.FEAT_TIME, FieldName.FEAT_AGE]
                + ([FieldName.FEAT_DYNAMIC_REAL]
                   if config.num_dynamic_real_features > 0 else []),
            ),
            RenameFields(mapping={
                FieldName.FEAT_STATIC_CAT:  'static_categorical_features',
                FieldName.FEAT_STATIC_REAL: 'static_real_features',
                FieldName.FEAT_TIME:        'time_features',
                FieldName.TARGET:           'values',
                FieldName.OBSERVED_VALUES:  'observed_mask',
            }),
        ]
    )


def create_instance_splitter(
    config: PretrainedConfig,
    mode: str,
    train_sampler: Optional[InstanceSampler] = None,
    validation_sampler: Optional[InstanceSampler] = None,
) -> Transformation:
    assert mode in ['train', 'validation', 'test']
    instance_sampler = {
        'train': train_sampler or ExpectedNumInstanceSampler(
            num_instances=1.0, min_future=config.prediction_length
        ),
        'validation': validation_sampler or ValidationSplitSampler(
            min_future=config.prediction_length
        ),
        'test': TestSplitSampler(),
    }[mode]

    return InstanceSplitter(
        target_field='values',
        is_pad_field=FieldName.IS_PAD,
        start_field=FieldName.START,
        forecast_start_field=FieldName.FORECAST_START,
        instance_sampler=instance_sampler,
        past_length=config.context_length + max(config.lags_sequence),
        future_length=config.prediction_length,
        time_series_fields=['time_features', 'observed_mask'],
    )

In [ ]:
def _prediction_fields(config):
    fields = [
        'past_time_features', 'past_values',
        'past_observed_mask', 'future_time_features',
    ]
    if config.num_static_categorical_features > 0:
        fields.append('static_categorical_features')
    if config.num_static_real_features > 0:
        fields.append('static_real_features')
    return fields


def create_train_dataloader(
    config, freq, data, batch_size, num_batches_per_epoch,
    shuffle_buffer_length=None, cache_data=True, **kwargs,
) -> Iterable:
    input_names   = _prediction_fields(config)
    training_names = input_names + ['future_values', 'future_observed_mask']

    transformation    = create_transformation(freq, config)
    transformed_data  = transformation.apply(data, is_train=True)
    if cache_data:
        transformed_data = Cached(transformed_data)

    instance_splitter = create_instance_splitter(config, 'train')
    stream            = Cyclic(transformed_data).stream()
    training_instances = instance_splitter.apply(stream)

    return as_stacked_batches(
        training_instances,
        batch_size=batch_size,
        shuffle_buffer_length=shuffle_buffer_length,
        field_names=training_names,
        output_type=torch.tensor,
        num_batches_per_epoch=num_batches_per_epoch,
    )


def create_backtest_dataloader(config, freq, data, batch_size, **kwargs):
    """Evaluation dataloader — miroir de l'original FAYAM."""
    input_names  = _prediction_fields(config)
    transformation   = create_transformation(freq, config)
    transformed_data = transformation.apply(data)
    instance_sampler = create_instance_splitter(config, 'validation')
    testing_instances = instance_sampler.apply(transformed_data, is_train=True)
    return as_stacked_batches(
        testing_instances,
        batch_size=batch_size,
        output_type=torch.tensor,
        field_names=input_names,
    )


def create_attention_dataloader(config, freq, data, batch_size=1):
    """Comme backtest, mais renvoie aussi future_values pour le forward teacher-forced."""
    fields = _prediction_fields(config) + ['future_values', 'future_observed_mask']
    transformation   = create_transformation(freq, config)
    transformed_data = transformation.apply(data)
    instance_sampler = create_instance_splitter(config, 'validation')
    testing_instances = instance_sampler.apply(transformed_data, is_train=True)
    return as_stacked_batches(
        testing_instances,
        batch_size=batch_size,
        output_type=torch.tensor,
        field_names=fields,
    )


print('Fonctions pipeline définies.')

## 6 — Modèle

In [ ]:
from transformers import TimeSeriesTransformerConfig, TimeSeriesTransformerForPrediction

config = TimeSeriesTransformerConfig(
    prediction_length=PREDICTION_LENGTH,
    context_length=CONTEXT_LENGTH,
    lags_sequence=lags_sequence,
    num_time_features=len(time_features) + 1,
    num_static_categorical_features=1,
    cardinality=[len(train_dataset)],
    embedding_dimension=EMBEDDING_DIM,
    encoder_layers=ENCODER_LAYERS,
    decoder_layers=DECODER_LAYERS,
    d_model=D_MODEL,
)

model  = TimeSeriesTransformerForPrediction(config)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f'Paramètres total    : {n_params:,}')
print(f'Distribution output : {model.config.distribution_output}')
print(f'Device              : {device}')

In [ ]:
train_dataloader = create_train_dataloader(
    config=config, freq=FREQ, data=train_dataset,
    batch_size=BATCH_SIZE_TRAIN, num_batches_per_epoch=NUM_BATCHES_EPOCH,
)
test_dataloader = create_backtest_dataloader(
    config=config, freq=FREQ, data=test_dataset,
    batch_size=BATCH_SIZE_TEST,
)

batch = next(iter(train_dataloader))
print('Shapes du premier batch :')
for k, v in batch.items():
    print(f'  {k:<40s} {str(v.shape):<25s} {v.dtype}')

In [ ]:
# Sanity check : forward pass avant entraînement
model.eval()
with torch.no_grad():
    _out = model(
        past_values=batch['past_values'].to(device),
        past_time_features=batch['past_time_features'].to(device),
        past_observed_mask=batch['past_observed_mask'].to(device),
        static_categorical_features=batch['static_categorical_features'].to(device)
            if config.num_static_categorical_features > 0 else None,
        static_real_features=batch['static_real_features'].to(device)
            if config.num_static_real_features > 0 else None,
        future_values=batch['future_values'].to(device),
        future_time_features=batch['future_time_features'].to(device),
        future_observed_mask=batch['future_observed_mask'].to(device),
    )
print(f'Loss initiale (avant entraînement) : {_out.loss.item():.4f}')

## 7 — Entraînement

- Gradient clipping à 1.0 (stabilité Transformer)
- Checkpoint toutes les `CHECKPOINT_EVERY` époques → Drive
- Reprise automatique depuis le dernier checkpoint si la cellule est ré-exécutée

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=LR, betas=BETAS, weight_decay=WEIGHT_DECAY)

# ── Reprise depuis checkpoint ────────────────────────────────────────────────
START_EPOCH  = 0
loss_history = []
ckpt_path    = f'{DRIVE_BASE}/checkpoints/latest.pt'

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    START_EPOCH  = ckpt['epoch'] + 1
    loss_history = ckpt.get('loss_history', [])
    print(f'Reprise depuis epoch {START_EPOCH}')
else:
    print('Démarrage from scratch')

# ── Boucle d entraînement ────────────────────────────────────────────────────
model.train()

for epoch in range(START_EPOCH, NUM_EPOCHS):
    epoch_losses = []
    pbar = tqdm(train_dataloader, desc=f'Epoch {epoch:03d}/{NUM_EPOCHS-1}', leave=False)

    for b in pbar:
        optimizer.zero_grad()
        out = model(
            static_categorical_features=b['static_categorical_features'].to(device)
                if config.num_static_categorical_features > 0 else None,
            static_real_features=b['static_real_features'].to(device)
                if config.num_static_real_features > 0 else None,
            past_time_features=b['past_time_features'].to(device),
            past_values=b['past_values'].to(device),
            future_time_features=b['future_time_features'].to(device),
            future_values=b['future_values'].to(device),
            past_observed_mask=b['past_observed_mask'].to(device),
            future_observed_mask=b['future_observed_mask'].to(device),
        )
        out.loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
        epoch_losses.append(out.loss.item())
        pbar.set_postfix({'loss': f'{out.loss.item():.4f}'})

    mean_loss = float(np.mean(epoch_losses))
    loss_history.append({'epoch': epoch, 'loss': mean_loss})
    print(f'Epoch {epoch:03d}  loss={mean_loss:.4f}')

    if (epoch + 1) % CHECKPOINT_EVERY == 0 or epoch == NUM_EPOCHS - 1:
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'loss_history': loss_history,
            'config': config.to_dict(),
        }, ckpt_path)
        print(f'  Checkpoint sauvegardé (epoch {epoch})')

print('Entraînement terminé.')

In [ ]:
epochs_log = [x['epoch'] for x in loss_history]
losses_log = [x['loss']  for x in loss_history]

plt.figure(figsize=(10, 3))
plt.plot(epochs_log, losses_log, linewidth=1.5)
plt.xlabel('Époque')
plt.ylabel('Loss NLL (moyenne par epoch)')
plt.title('Courbe de perte — baseline FAYAM TimeSeriesTransformer')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/results/loss_curve.png', dpi=150)
plt.show()

## 8 — Inférence

In [ ]:
model.eval()
forecasts = []

with torch.no_grad():
    for b in tqdm(test_dataloader, desc='Inférence'):
        out = model.generate(
            static_categorical_features=b['static_categorical_features'].to(device)
                if config.num_static_categorical_features > 0 else None,
            static_real_features=b['static_real_features'].to(device)
                if config.num_static_real_features > 0 else None,
            past_time_features=b['past_time_features'].to(device),
            past_values=b['past_values'].to(device),
            future_time_features=b['future_time_features'].to(device),
            past_observed_mask=b['past_observed_mask'].to(device),
        )
        forecasts.append(out.sequences.cpu().numpy())

forecasts = np.vstack(forecasts)
print(f'Shape forecasts : {forecasts.shape}  (séries, samples, horizon)')

## 9 — Métriques

In [ ]:
from evaluate import load as load_metric
from gluonts.time_feature import get_seasonality

mase_metric  = load_metric('evaluate-metric/mase')
smape_metric = load_metric('evaluate-metric/smape')

forecast_median = np.median(forecasts, axis=1)  # (n_series, prediction_length)

mase_vals  = []
smape_vals = []

for item_id, ts in enumerate(tqdm(test_dataset, desc='MASE/sMAPE')):
    training_data = ts['target'][:-PREDICTION_LENGTH]
    ground_truth  = ts['target'][-PREDICTION_LENGTH:]

    mase_vals.append(mase_metric.compute(
        predictions=forecast_median[item_id],
        references=np.array(ground_truth),
        training=np.array(training_data),
        periodicity=get_seasonality(FREQ),
    )['mase'])

    smape_vals.append(smape_metric.compute(
        predictions=forecast_median[item_id],
        references=np.array(ground_truth),
    )['smape'])

print(f'MASE  moyen : {np.mean(mase_vals):.4f}')
print(f'sMAPE moyen : {np.mean(smape_vals):.4f}')

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score, explained_variance_score
from scipy.stats import spearmanr


def compute_metrics(pred: np.ndarray, actual: np.ndarray) -> dict:
    rmse  = float(np.sqrt(mean_squared_error(actual, pred)))
    r2    = float(r2_score(actual, pred))
    spear = float(spearmanr(actual, pred).statistic)
    ev    = float(explained_variance_score(actual, pred))
    nrmse = rmse / (float(actual.max()) - float(actual.min()) + 1e-8)
    mape  = float(np.mean(np.abs((actual - pred) / (np.abs(actual) + 1e-8))))
    return {'RMSE': rmse, 'NRMSE': nrmse, 'R2': r2,
            'Spearman': spear, 'MAPE': mape, 'ExplainedVariance': ev}


all_metrics = []
for ts_index in range(len(test_dataset)):
    pred   = forecast_median[ts_index]
    actual = np.array(test_dataset[ts_index]['target'][-PREDICTION_LENGTH:])
    m = compute_metrics(pred, actual)
    m['ts_index']    = ts_index
    m['cluster']     = test_dataset[ts_index]['cluster']
    m['function_id'] = test_dataset[ts_index]['function_id']
    m['MASE']        = mase_vals[ts_index]
    m['sMAPE']       = smape_vals[ts_index]
    all_metrics.append(m)

df_metrics = pd.DataFrame(all_metrics)
print('\nRésumé statistique (toutes séries) :')
print(df_metrics[['RMSE', 'R2', 'Spearman', 'sMAPE', 'MASE']].describe().round(4))

In [ ]:
results_summary = {
    'run_name':   RUN_NAME,
    'seed':       SEED,
    'num_epochs': NUM_EPOCHS,
    'n_series':   len(test_dataset),
    'global': {
        'MASE_mean':     float(np.mean(mase_vals)),
        'sMAPE_mean':    float(np.mean(smape_vals)),
        'RMSE_mean':     float(df_metrics['RMSE'].mean()),
        'R2_mean':       float(df_metrics['R2'].mean()),
        'Spearman_mean': float(df_metrics['Spearman'].mean()),
    },
    'per_series': df_metrics.to_dict(orient='records'),
}

with open(f'{DRIVE_BASE}/results/metrics.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
df_metrics.to_csv(f'{DRIVE_BASE}/results/metrics.csv', index=False)

print('Métriques sauvegardées.')
print('\n── Résumé global ──')
for k, v in results_summary['global'].items():
    print(f'  {k:<22s} : {v:.4f}')

In [ ]:
# Synthèse par cluster — comparaison FAYAM par profil de charge
cluster_agg = (df_metrics
    .groupby('cluster')
    .agg(
        n=('ts_index', 'count'),
        MASE=('MASE', 'mean'),
        sMAPE=('sMAPE', 'mean'),
        RMSE=('RMSE', 'mean'),
        R2=('R2', 'mean'),
        Spearman=('Spearman', 'mean'),
    )
    .round(4)
)
print('── Métriques moyennes par cluster ──')
print(cluster_agg)
cluster_agg.to_csv(f'{DRIVE_BASE}/results/metrics_by_cluster.csv')

# Mapping ts_index ↔ cluster ↔ function_id (pour réutilisation H1/H3)
cluster_mapping = pd.DataFrame([
    {'ts_index': i, 'cluster': s['cluster'], 'function_id': s['function_id']}
    for i, s in enumerate(all_series)
])
cluster_mapping.to_csv(f'{DRIVE_BASE}/results/cluster_mapping.csv', index=False)
print('\nFichiers sauvegardés : metrics_by_cluster.csv + cluster_mapping.csv')

## 10 — Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].scatter(mase_vals, smape_vals, alpha=0.4, s=20)
axes[0].set_xlabel('MASE')
axes[0].set_ylabel('sMAPE')
axes[0].set_title('MASE vs sMAPE (par série)')
axes[0].grid(alpha=0.3)

axes[1].scatter(df_metrics['R2'], df_metrics['Spearman'], alpha=0.4, s=20, color='darkorange')
axes[1].set_xlabel('R²')
axes[1].set_ylabel('Spearman')
axes[1].set_title('R² vs Spearman (par série)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/results/scatter_metrics.png', dpi=150)
plt.show()

In [ ]:
def plot_forecast(ts_index, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(12, 3))
    index = pd.period_range(
        start=test_dataset[ts_index][FieldName.START],
        periods=len(test_dataset[ts_index][FieldName.TARGET]),
        freq=FREQ,
    ).to_timestamp()

    ax.plot(index[-2 * PREDICTION_LENGTH:],
            test_dataset[ts_index]['target'][-2 * PREDICTION_LENGTH:],
            label='Réel', linewidth=1)
    ax.plot(index[-PREDICTION_LENGTH:],
            forecast_median[ts_index],
            label='Médiane prévue', alpha=0.85, linewidth=1)
    ax.fill_between(
        index[-PREDICTION_LENGTH:],
        forecasts[ts_index].mean(0) - forecasts[ts_index].std(0),
        forecasts[ts_index].mean(0) + forecasts[ts_index].std(0),
        alpha=0.25, label='±1 std',
    )
    rmse_val = df_metrics.loc[df_metrics['ts_index'] == ts_index, 'RMSE'].values[0]
    r2_val   = df_metrics.loc[df_metrics['ts_index'] == ts_index, 'R2'].values[0]
    ax.set_title(f'Série {ts_index}  RMSE={rmse_val:.2f}  R²={r2_val:.3f}')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)


n_plot = min(6, len(test_dataset))
fig, axes = plt.subplots(n_plot, 1, figsize=(14, 3 * n_plot), squeeze=False)
for i, ax in enumerate(axes.flatten()):
    plot_forecast(i, ax)
plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/results/forecast_samples.png', dpi=120)
plt.show()

## 11 — Extraction attention (`output_attentions=True`)

Extraction des poids d'attention post-entraînement pour préparer les phases XAI (H1/H3).

**Stratégie** : forward pass teacher-forcé (on fournit `future_values` = vérité terrain) sur les `MAX_ATTN_SAMPLES` premières séries du test set. On ne ré-entraîne pas.

**Ce qu'on sauvegarde** :
- `cross_attentions_last_layer.npy` — shape `(n, heads, pred_len, ctx_len)` : le décodeur qui attend l'encodeur → carte principale pour XAI
- `encoder_attentions_last_layer.npy` — shape `(n, heads, ctx_len, ctx_len)` : auto-attention encodeur

> Ces tenseurs serviront de **sanity check** pour comparer avec la carte d'évidence SoftCAM (H1) : si les deux sont cohérentes, la méthode est validée.

In [ ]:
attn_loader = create_attention_dataloader(
    config=config, freq=FREQ, data=test_dataset, batch_size=1,
)

model.eval()
cross_attn_list = []
enc_attn_list   = []

n_samples = min(MAX_ATTN_SAMPLES, len(test_dataset))

with torch.no_grad():
    for i, batch in enumerate(tqdm(attn_loader, total=n_samples, desc='Extraction attention')):
        if i >= n_samples:
            break
        out = model(
            past_values=batch['past_values'].to(device),
            past_time_features=batch['past_time_features'].to(device),
            past_observed_mask=batch['past_observed_mask'].to(device),
            future_values=batch['future_values'].to(device),
            future_time_features=batch['future_time_features'].to(device),
            future_observed_mask=batch['future_observed_mask'].to(device),
            static_categorical_features=batch['static_categorical_features'].to(device)
                if config.num_static_categorical_features > 0 else None,
            output_attentions=True,
        )
        # cross_attentions : tuple (decoder_layers,) tenseurs shape (1, heads, pred_len, ctx_len)
        cross_attn_list.append(out.cross_attentions[-1].squeeze(0).cpu().numpy())
        enc_attn_list.append(out.encoder_attentions[-1].squeeze(0).cpu().numpy())

cross_attn_arr = np.stack(cross_attn_list)   # (n, heads, pred_len, ctx_len)
enc_attn_arr   = np.stack(enc_attn_list)     # (n, heads, ctx_len, ctx_len)

np.save(f'{DRIVE_BASE}/attentions/cross_attentions_last_layer.npy', cross_attn_arr)
np.save(f'{DRIVE_BASE}/attentions/encoder_attentions_last_layer.npy', enc_attn_arr)

print(f'cross_attn_arr : {cross_attn_arr.shape}  → (séries, têtes, pred_len, ctx_len)')
print(f'enc_attn_arr   : {enc_attn_arr.shape}  → (séries, têtes, ctx_len, ctx_len)')
print(f'Sauvegardés dans {DRIVE_BASE}/attentions/')

In [ ]:
# Carte d attention moyenne (moyenne séries + têtes) — vérification visuelle
mean_cross = cross_attn_arr.mean(axis=(0, 1))   # (pred_len, ctx_len)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

im0 = axes[0].imshow(mean_cross, aspect='auto', cmap='viridis', origin='lower')
plt.colorbar(im0, ax=axes[0])
axes[0].set_xlabel('Contexte passé (tokens)')
axes[0].set_ylabel('Horizon prédit (pas)')
axes[0].set_title(f'Cross-attention moyenne — couche {DECODER_LAYERS}\n({cross_attn_arr.shape[0]} séries, moy. sur {cross_attn_arr.shape[1]} têtes)')

# Profil temporel : quel pas du passé est le plus regardé ?
attn_profile = mean_cross.mean(axis=0)  # (ctx_len,)
axes[1].plot(attn_profile, linewidth=1.2)
axes[1].set_xlabel('Token de contexte (plus récent = droite)')
axes[1].set_ylabel('Poids moyen')
axes[1].set_title('Profil temporel — contexte passé le plus regardé')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/attentions/cross_attention_heatmap.png', dpi=150)
plt.show()

In [ ]:
# Carte d attention moyenne (moyenne séries + têtes) — vérification visuelle
mean_cross = cross_attn_arr.mean(axis=(0, 1))   # (pred_len, ctx_len)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))

im0 = axes[0].imshow(mean_cross, aspect='auto', cmap='viridis', origin='lower')
plt.colorbar(im0, ax=axes[0])
axes[0].set_xlabel('Contexte passé (tokens)')
axes[0].set_ylabel('Horizon prédit (pas)')
axes[0].set_title(f'Cross-attention moyenne — couche {DECODER_LAYERS}\n({MAX_ATTN_SAMPLES} séries, moy. sur {cross_attn_arr.shape[1]} têtes)')

# Profil temporel : quel pas du passé est le plus regardé ?
attn_profile = mean_cross.mean(axis=0)  # (ctx_len,)
axes[1].plot(attn_profile, linewidth=1.2)
axes[1].set_xlabel('Token de contexte (plus récent = droite)')
axes[1].set_ylabel('Poids moyen')
axes[1].set_title('Profil temporel — contexte passé le plus regardé')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_BASE}/attentions/cross_attention_heatmap.png', dpi=150)
plt.show()

## 12 — Sauvegarde finale

In [ ]:
# Modèle final
torch.save(model.state_dict(), f'{DRIVE_BASE}/checkpoints/model_final.pt')

# Métadonnées du run
run_meta = {
    'run_name':     RUN_NAME,
    'seed':         SEED,
    'num_epochs':   NUM_EPOCHS,
    'torch_version': torch.__version__,
    'device':       str(device),
    'config':       config.to_dict(),
    'loss_history': loss_history,
}
with open(f'{DRIVE_BASE}/run_meta.json', 'w') as f:
    json.dump(run_meta, f, indent=2)

print('=' * 60)
print(f'Run complet. Artefacts dans :')
print(f'  {DRIVE_BASE}')
print('  checkpoints/  model_final.pt + latest.pt')
print('  results/      metrics.json + metrics.csv + figures')
print('  attentions/   *.npy + heatmap')
print('=' * 60)
print('\n── Résumé métriques finales ──')
for k, v in results_summary['global'].items():
    print(f'  {k:<22s} : {v:.4f}')